# Ejemplo 2 · Clasificar gastos por texto
**Estud-IA · Clase 5 · Intermedio.**

**Problema.** Tienes descripciones de gastos («Uber al aeropuerto») y quieres que el computador les ponga la categoría.

**El modelo, en palabras simples (TF-IDF + regresión logística).**
- **TF-IDF** convierte cada texto en números: cuenta las palabras y da más peso a las que distinguen (como «Uber») y menos a las comunes (como «de»). Es la misma idea de *tokenizar* de la clase 4: texto a números.
- **Regresión logística** aprende, con ejemplos etiquetados, cuántos puntos suma cada palabra a cada categoría. La categoría con más puntos gana. No entiende el significado: reconoce patrones de palabras.

**Datos.** Los gastos de este notebook son **ficticios**, generados con plantillas, errores de digitación y algunas etiquetas dudosas para que se parezcan a los reales. Pon tu CSV propio en la celda de datos cuando quieras (columnas `descripcion`, `categoria`, sin información sensible).

## Parte 0 · Datos

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(1540)

base = {
 "transporte":      ["uber al aeropuerto","taxi a la oficina","gasolina del carro","peaje autopista norte","pasaje de bus","parqueadero centro","metro tarjeta recarga","taxi nocturno"],
 "alimentación":    ["almuerzo con cliente","mercado semanal","cena de equipo","desayuno reunión","domicilio de comida","cafe con proveedor","restaurante con cliente","snacks oficina"],
 "tecnología":      ["licencia de software","suscripción de nube","cable usb nuevo","mouse inalámbrico","hosting de la página","dominio web anual","teclado mecánico","disco duro externo"],
 "salud":           ["consulta médica","medicamentos farmacia","examen de laboratorio","seguro de salud","gafas formuladas","cita odontológica","vacuna anual","terapia física"],
 "entretenimiento": ["entradas al cine","plan de streaming","concierto fin de semana","videojuego nuevo","paseo de integración","boletas del teatro","suscripción de música","juegos de mesa"],
 "servicios":       ["factura de energía","recibo del agua","internet del mes","arriendo de oficina","plan de celular","servicio de aseo","gas natural","administración edificio"],
}
extra = ["", " mes de marzo", " pagado hoy", " factura 123", " urgente", " viaje", " equipo", " semana 3"]
def ruido(t):
    t = list(t)
    if len(t) > 5 and rng.random() < 0.25:
        j = int(rng.integers(1, len(t) - 1)); t[j] = t[j-1]
    return "".join(t)

filas = []
for cat, frases in base.items():
    for _ in range(100):
        f = frases[int(rng.integers(len(frases)))] + extra[int(rng.integers(len(extra)))]
        filas.append((ruido(f), cat))
df = pd.DataFrame(filas, columns=["descripcion", "categoria"])

# etiquetas dudosas (como pasa en la vida real): 5 % con otra categoría
cats = list(base)
mal = rng.choice(len(df), size=int(0.05 * len(df)), replace=False)
for i in mal: df.loc[i, "categoria"] = cats[int(rng.integers(len(cats)))]

# Para usar TUS datos: df = pd.read_csv("mis_gastos.csv")  (columnas descripcion, categoria)
print(df.shape); display(df.sample(8, random_state=1))
print(df.categoria.value_counts())

## Parte 1 · Separar y entrenar
Se separa una parte que el modelo **nunca ve al aprender**. Aquí hay entrenamiento y prueba: la prueba se abre una sola vez, al final.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

Xe, Xp, ye, yp = train_test_split(df.descripcion, df.categoria, test_size=0.25, random_state=1540, stratify=df.categoria)
modelo = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
modelo.fit(Xe, ye)
pred = modelo.predict(Xp)
k = int((pred == yp.values).sum())
print(f"Aciertos en datos de prueba: {k} de {len(yp)}")

## Parte 2 · Qué aprendió (las palabras que más pesan)
Mira las 5 palabras que más suman a cada categoría. Si ves algo raro aquí, ese es el primer lugar donde sospechar.

In [ ]:
tf = modelo.named_steps["tfidfvectorizer"]; lr = modelo.named_steps["logisticregression"]
vocab = np.array(tf.get_feature_names_out())
for i, c_ in enumerate(lr.classes_):
    top = vocab[np.argsort(lr.coef_[i])[::-1][:5]]
    print(f"{c_:16s} ->", ", ".join(top))

## Parte 3 · Desconfía de un número perfecto
Cuando los datos son repetitivos, el acierto sale altísimo y engaña. La prueba honesta es con **frases nuevas escritas por ti**. Aquí van 10 de ejemplo, con la categoría que tú esperarías. **Cámbialas por las tuyas.**

In [ ]:
nuevas = [
 ("pago del parqueadero del aeropuerto","transporte"),
 ("almuerzo de equipo en el centro","alimentación"),
 ("renovación de la licencia del programa de diseño","tecnología"),
 ("copago de la consulta con el especialista","salud"),
 ("boletas para el partido","entretenimiento"),
 ("recibo de la luz","servicios"),
 ("gimnasio mensualidad","salud"),
 ("curso en línea de excel","tecnología"),
 ("regalo para el cliente","entretenimiento"),
 ("lavado del carro","transporte"),
]
textos = [t for t, _ in nuevas]; reales = [c_ for _, c_ in nuevas]
pn = modelo.predict(textos)
tabla = pd.DataFrame({"frase": textos, "yo esperaba": reales, "el modelo dice": pn, "¿acierta?": [a == b for a, b in zip(reales, pn)]})
display(tabla)
print(f"Frases nuevas: {int(tabla['¿acierta?'].sum())} de {len(tabla)}")

Mira los fallos y las frases ambiguas («gimnasio»: ¿salud o entretenimiento?). Una etiqueta ambigua es un problema de **datos**, no del modelo: si entra basura, sale basura.

## Parte 4 · Un solo cambio, y se vuelve a medir
**Cambio:** usar pares de palabras (bigramas) además de palabras sueltas. Todo lo demás queda igual, incluida la partición.

In [ ]:
modelo2 = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=2000)).fit(Xe, ye)
k2 = int((modelo2.predict(Xp) == yp.values).sum())
n2 = int(sum(a == b for a, b in zip(reales, modelo2.predict(textos))))
print(f"Prueba:        antes {k} de {len(yp)}  ->  ahora {k2} de {len(yp)}")
print(f"Frases nuevas: antes {int(tabla['¿acierta?'].sum())} de {len(tabla)}  ->  ahora {n2} de {len(tabla)}")
print("Regla de la clase: el cambio vale si gana por el margen que escribiste ANTES de ejecutar.")

## Parte 5 · Tu turno: tu ficha convertida en prompt
Ficha (ejemplo): *Camino B. Cambio: bigramas. Medida: k de N en frases nuevas. Criterio: lo adopto si gana por 2 o más de 10. Riesgo: etiquetas ambiguas, categorías desiguales.*

Prompt para tu IA:
```
Tengo gastos con las columnas descripcion y categoria (ejemplo de fila: "uber al aeropuerto", "transporte").
Hazme un notebook de Colab con TF-IDF y regresión logística que separe datos que el modelo no vea,
me diga cuántos acertó de cuántos, me muestre las palabras que más pesan por categoría
y una tabla con los errores. Si no tengo datos, genera 600 filas ficticias.
Explica cada celda sin matemáticas. No cambies nada más de lo que te pedí.
```
Entrega: tu resultado en «k de N», 10 frases escritas por ti y el único cambio que probarías después.